In [1]:
!nvidia-smi
!python --version

Thu Apr 16 20:32:04 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA A100-SXM4-40GB          Off |   00000000:00:04.0 Off |                    0 |
| N/A   30C    P0             40W /  400W |       0MiB /  40960MiB |      0%      Default |
|                                         |                        |             Disabled |
+-----------------------------------------+-----

In [2]:
!pip -q uninstall -y torch torchvision torchaudio vllm
!pip -q install -U uv
!uv pip install lmcache vllm --torch-backend=auto
!pip -q install transformers accelerate bitsandbytes fastapi uvicorn psutil GPUtil

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.9/24.9 MB 101.3 MB/s eta 0:00:00
Using Python 3.12.13 environment at: /usr
Resolved 189 packages in 2.12s
Prepared 84 packages in 28.20s
Uninstalled 8 packages in 404ms
Installed 84 packages in 286ms
 + aiofile==3.9.0
 + anthropic==0.96.0
 + apache-tvm-ffi==0.1.10
 + astor==0.8.1
 + awscrt==0.32.0
 + blake3==1.0.8
 + caio==0.9.25
 + cbor2==5.9.0
 + compressed-tensors==0.14.0.1
 - cuda-bindings==12.9.4
 + cuda-bindings==13.0.3
 - cuda-python==12.9.4
 + cuda-python==13.0.3
 + cufile-python==0.2.0
 + depyf==0.20.0
 + diskcache==5.6.3
 + dnspython==2.8.0
 + email-validator==2.3.0
 + fastapi-cli==0.0.24
 + fastapi-cloud-cli==0.17.0
 + fastar==0.11.0
 + flashinfer-cubin==0.6.6
 + flashinfer-python==0.6.6
 + gguf==0.18.0
 - huggingface-hub==1.10.1
 + huggingface-hub==0.36.2
 + ijson==3.5.0
 + interegular==0.3.3
 + jmespath==1.1.0
 - lark==1.3.1
 + lark==1.2.2
 + llguidance==1.3.0
 - llvmlite==0.43.0
 + llvmlite==0.44.0
 + lm-format-enforcer==0.11

In [10]:
import torch
import vllm
import transformers
import lmcache


print("PyTorch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())
print("GPU count:", torch.cuda.device_count())
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "None")
print("vLLM:", vllm.__version__)
print("Transformers:", transformers.__version__)

PyTorch: 2.10.0+cu130
CUDA available: True
GPU count: 1
GPU: NVIDIA A100-SXM4-40GB
vLLM: 0.19.0
Transformers: 4.57.6


In [4]:
# !nohup vllm serve Qwen/Qwen3-8B \
#   --dtype bfloat16 \
#   --max-model-len 8192 \
#   --gpu-memory-utilization 0.9 \

!nohup vllm serve Qwen/Qwen3-8B \
  --dtype bfloat16 \
  --max-model-len 8192 \
  --gpu-memory-utilization 0.9 \
  --kv-offloading-backend lmcache \
  --kv-offloading-size 8 \
  --disable-hybrid-kv-cache-manager \
  > vllm.log 2>&1 &

In [9]:
!tail -n 200 vllm.log

2026-04-16 20:35:39.627738: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2026-04-16 20:35:39.646049: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1776371739.668998    2415 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1776371739.676244    2415 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1776371739.694071    2415 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking 

In [15]:
!curl http://localhost:8000/v1/completions -H "Content-Type: application/json" -d '{"model": "Qwen/Qwen3-8B", "prompt": "Qwen3 is the latest generation of large language models in Qwen series, offering a comprehensive suite of dense and mixture-of-experts", "max_tokens": 100, "temperature": 0.7}'

{"id":"cmpl-9f6585f419ba1a81","object":"text_completion","created":1776372085,"model":"Qwen/Qwen3-8B","choices":[{"index":0,"text":" (MoE) architectures. It is built upon a robust foundation of the Qwen2 model, incorporating advanced techniques such as enhanced training data, optimized model architecture, and improved training strategies. Qwen3 provides a more powerful and efficient large language model, capable of handling complex tasks with high accuracy and performance.\n\nThe Qwen3 series includes multiple models with different parameters, such as Qwen3, Qwen3-Chat, Qwen3-72B, and Qwen3-7","logprobs":null,"finish_reason":"length","stop_reason":null,"token_ids":null,"prompt_logprobs":null,"prompt_token_ids":null}],"service_tier":null,"system_fingerprint":null,"usage":{"prompt_tokens":27,"total_tokens":127,"completion_tokens":100,"prompt_tokens_details":null},"kv_transfer_params":null}

In [16]:
import requests

url = "http://localhost:8000/v1/chat/completions"

payload = {
    "model": "Qwen/Qwen3-8B",
    "messages": [
        {"role": "system", "content": "You are a helpful assistant."},
        {"role": "user", "content": "Write 3 short bullet points about A100 GPUs."}
    ],
    "temperature": 0.7,
    "max_tokens": 128
}

try:
    resp = requests.post(url, json=payload, timeout=60)
    resp.raise_for_status()

    data = resp.json()
    print(resp.status_code)
    print(data)
    print(data["choices"][0]["message"]["content"])

except Exception as e:
    print("ERROR:", e)

200
{'id': 'chatcmpl-b95485df724ea577', 'object': 'chat.completion', 'created': 1776372123, 'model': 'Qwen/Qwen3-8B', 'choices': [{'index': 0, 'message': {'role': 'assistant', 'content': "<think>\nOkay, the user wants three short bullet points about A100 GPUs. Let me start by recalling what I know about the A100. It's part of NVIDIA's A100 series, right? I remember it's a high-performance GPU used in data centers. The first point should highlight its key feature. Maybe mention the architecture, like Ampere. Then, the second point could be about its applications, such as AI training and high-performance computing. The third point might focus on the memory capacity, like 80GB or 40GB variants. Wait, I should check if the A", 'refusal': None, 'annotations': None, 'audio': None, 'function_call': None, 'tool_calls': [], 'reasoning': None}, 'logprobs': None, 'finish_reason': 'length', 'stop_reason': None, 'token_ids': None}], 'service_tier': None, 'system_fingerprint': None, 'usage': {'promp

In [17]:
import time
import json
import statistics
import requests

BASE_URL = "http://localhost:8000/v1"
MODEL = "Qwen/Qwen3-8B"

In [18]:
def run_inference(prompt, max_tokens=128, temperature=0.7):
    payload = {
        "model": MODEL,
        "messages": [
            {"role": "system", "content": "You are a helpful assistant."},
            {"role": "user", "content": prompt},
        ],
        "temperature": temperature,
        "max_tokens": max_tokens,
    }

    start = time.time()
    resp = requests.post(f"{BASE_URL}/chat/completions", json=payload, timeout=120)
    resp.raise_for_status()
    latency = time.time() - start

    data = resp.json()
    usage = data.get("usage", {})
    content = data["choices"][0]["message"]["content"]

    completion_tokens = usage.get("completion_tokens", 0)
    prompt_tokens = usage.get("prompt_tokens", 0)
    total_tokens = usage.get("total_tokens", prompt_tokens + completion_tokens)

    metrics = {
        "latency_sec": round(latency, 3),
        "prompt_tokens": prompt_tokens,
        "completion_tokens": completion_tokens,
        "total_tokens": total_tokens,
        "tokens_per_sec": round(completion_tokens / latency, 2) if latency > 0 else 0.0,
    }

    return content, metrics, data

In [19]:
def benchmark_prompts(prompts, runs=3, max_tokens=128, temperature=0.7):
    results = []

    for run in range(runs):
        print(f"Run {run + 1}/{runs}")
        for i, prompt in enumerate(prompts, 1):
            content, metrics, _ = run_inference(
                prompt,
                max_tokens=max_tokens,
                temperature=temperature,
            )
            results.append({
                "run": run + 1,
                "prompt_index": i,
                "prompt": prompt,
                **metrics,
            })
            print(
                f"  Prompt {i}: "
                f"{metrics['latency_sec']:.3f}s, "
                f"{metrics['completion_tokens']} output tokens, "
                f"{metrics['tokens_per_sec']:.2f} tok/s"
            )

    summary = {
        "avg_latency_sec": round(statistics.mean(r["latency_sec"] for r in results), 3),
        "avg_tokens_per_sec": round(statistics.mean(r["tokens_per_sec"] for r in results), 2),
        "avg_completion_tokens": round(statistics.mean(r["completion_tokens"] for r in results), 2),
        "num_requests": len(results),
    }

    return results, summary


In [20]:
def show_one_test(prompt):
    content, metrics, data = run_inference(prompt)
    print("Reply:\n")
    print(content)
    print("\nMetrics:\n")
    print(json.dumps(metrics, indent=2))
    return data

In [21]:
show_one_test("Write 3 short bullet points about A100 GPUs.")

Reply:

<think>
Okay, the user asked for three short bullet points about A100 GPUs. Let me start by recalling what I know about the A100. It's part of NVIDIA's A-series, right? So first, I should mention its key features. The A100 has a high number of CUDA cores, which is important for parallel processing. Maybe mention the 80 billion transistors? That's a big number and shows advanced technology.

Next, the memory. The A100 has a large amount of memory, like 40GB or 80GB. Wait, I think

Metrics:

{
  "latency_sec": 1.731,
  "prompt_tokens": 32,
  "completion_tokens": 128,
  "total_tokens": 160,
  "tokens_per_sec": 73.93
}


{'id': 'chatcmpl-8180144fd3de1ce0',
 'object': 'chat.completion',
 'created': 1776372226,
 'model': 'Qwen/Qwen3-8B',
 'choices': [{'index': 0,
   'message': {'role': 'assistant',
    'content': "<think>\nOkay, the user asked for three short bullet points about A100 GPUs. Let me start by recalling what I know about the A100. It's part of NVIDIA's A-series, right? So first, I should mention its key features. The A100 has a high number of CUDA cores, which is important for parallel processing. Maybe mention the 80 billion transistors? That's a big number and shows advanced technology.\n\nNext, the memory. The A100 has a large amount of memory, like 40GB or 80GB. Wait, I think",
    'refusal': None,
    'annotations': None,
    'audio': None,
    'function_call': None,
    'tool_calls': [],
    'reasoning': None},
   'logprobs': None,
   'finish_reason': 'length',
   'stop_reason': None,
   'token_ids': None}],
 'service_tier': None,
 'system_fingerprint': None,
 'usage': {'prompt_tokens':

In [22]:
test_prompts = [
    "Write 3 short bullet points about A100 GPUs.",
    "Explain what tensor parallelism is in 3 sentences.",
    "Give a short summary of vLLM.",
]

results, summary = benchmark_prompts(test_prompts, runs=3, max_tokens=128)
print(json.dumps(summary, indent=2))

Run 1/3
  Prompt 1: 1.722s, 128 output tokens, 74.32 tok/s
  Prompt 2: 1.721s, 128 output tokens, 74.36 tok/s
  Prompt 3: 1.722s, 128 output tokens, 74.33 tok/s
Run 2/3
  Prompt 1: 1.721s, 128 output tokens, 74.36 tok/s
  Prompt 2: 1.721s, 128 output tokens, 74.36 tok/s
  Prompt 3: 1.721s, 128 output tokens, 74.36 tok/s
Run 3/3
  Prompt 1: 1.721s, 128 output tokens, 74.37 tok/s
  Prompt 2: 1.721s, 128 output tokens, 74.38 tok/s
  Prompt 3: 1.721s, 128 output tokens, 74.37 tok/s
{
  "avg_latency_sec": 1.721,
  "avg_tokens_per_sec": 74.36,
  "avg_completion_tokens": 128,
  "num_requests": 9
}


### AWQ 4-bit

In [34]:
!pkill -f "vllm serve"
!nohup vllm serve Qwen/Qwen3-8B-AWQ \
  --dtype auto \
  --max-model-len 2048 \
  --gpu-memory-utilization 0.2 \
  --kv-offloading-backend lmcache \
  --kv-offloading-size 8 \
  --disable-hybrid-kv-cache-manager \
  > vllm.log 2>&1 &

In [42]:
!tail -n 200 vllm.log

# !sleep 10 && tail -n 100 vllm.log

2026-04-16 20:50:50.657762: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2026-04-16 20:50:50.676986: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1776372650.700069    7081 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1776372650.707732    7081 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1776372650.726299    7081 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking 

In [43]:
BASE_URL = "http://localhost:8000/v1"
MODEL = "Qwen/Qwen3-8B-AWQ"

In [44]:
show_one_test("Write 3 short bullet points about A100 GPUs.")

Reply:

<think>
Okay, the user wants three short bullet points about A100 GPUs. Let me start by recalling what I know about A100. First, the A100 is part of NVIDIA's H100 series, right? Wait, no, actually, the A100 is part of the A1000 series. Wait, maybe I should check that. Wait, no, the A100 is part of the A1000 series, but I think the H100 is the newer one. Wait, maybe I'm mixing up the generations. Let me make

Metrics:

{
  "latency_sec": 0.976,
  "prompt_tokens": 32,
  "completion_tokens": 128,
  "total_tokens": 160,
  "tokens_per_sec": 131.12
}


{'id': 'chatcmpl-aeabfe68bae715aa',
 'object': 'chat.completion',
 'created': 1776372797,
 'model': 'Qwen/Qwen3-8B-AWQ',
 'choices': [{'index': 0,
   'message': {'role': 'assistant',
    'content': "<think>\nOkay, the user wants three short bullet points about A100 GPUs. Let me start by recalling what I know about A100. First, the A100 is part of NVIDIA's H100 series, right? Wait, no, actually, the A100 is part of the A1000 series. Wait, maybe I should check that. Wait, no, the A100 is part of the A1000 series, but I think the H100 is the newer one. Wait, maybe I'm mixing up the generations. Let me make",
    'refusal': None,
    'annotations': None,
    'audio': None,
    'function_call': None,
    'tool_calls': [],
    'reasoning': None},
   'logprobs': None,
   'finish_reason': 'length',
   'stop_reason': None,
   'token_ids': None}],
 'service_tier': None,
 'system_fingerprint': None,
 'usage': {'prompt_tokens': 32,
  'total_tokens': 160,
  'completion_tokens': 128,
  'prompt_token

In [45]:
test_prompts = [
    "Write 3 short bullet points about A100 GPUs.",
    "Explain what tensor parallelism is in 3 sentences.",
    "Give a short summary of vLLM.",
]

results, summary = benchmark_prompts(test_prompts, runs=3, max_tokens=128)
print(json.dumps(summary, indent=2))

Run 1/3
  Prompt 1: 0.841s, 128 output tokens, 152.20 tok/s
  Prompt 2: 0.817s, 128 output tokens, 156.76 tok/s
  Prompt 3: 0.814s, 128 output tokens, 157.17 tok/s
Run 2/3
  Prompt 1: 0.815s, 128 output tokens, 157.02 tok/s
  Prompt 2: 0.814s, 128 output tokens, 157.16 tok/s
  Prompt 3: 0.814s, 128 output tokens, 157.27 tok/s
Run 3/3
  Prompt 1: 0.814s, 128 output tokens, 157.26 tok/s
  Prompt 2: 0.815s, 128 output tokens, 157.10 tok/s
  Prompt 3: 0.815s, 128 output tokens, 157.00 tok/s
{
  "avg_latency_sec": 0.818,
  "avg_tokens_per_sec": 156.55,
  "avg_completion_tokens": 128,
  "num_requests": 9
}
